In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '/app')
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List
from backtesting import Backtest

from strat.s_rsi_mr import (
    build_features,
    convert_to_ohlcv,
    RSIMeanReversionStrategy,
)
from core.enums import (
    g_open_col,
    g_high_col,
    g_low_col,
    g_close_col,
    g_volume_col,
    g_index_col,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

%matplotlib inline

/usr/local/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
BUNDLE_PATH = '/data/bundle/etf_x_list_features_bundle.parquet'

df_bundle = pd.read_parquet(BUNDLE_PATH)
print(f"Bundle shape: {df_bundle.shape}")
print(f"Bundle columns: {df_bundle.columns.tolist()[:10]}...")

def get_etf_symbols(df: pd.DataFrame) -> List[str]:
    symbols = set()
    for col in df.columns:
        if '_S_close_f32' in col:
            symbols.add(col.split('_')[0])
    return sorted(symbols)

etf_symbols = get_etf_symbols(df_bundle)
print(f"ETFs found: {etf_symbols}")

Bundle shape: (37946, 171)
Bundle columns: ['QQQ_S_open_f32', 'QQQ_S_high_f32', 'QQQ_S_low_f32', 'QQQ_S_close_f32', 'QQQ_S_volume_f64', 'QQQ_S_open_time_i', 'QQQ_S_close_time_i', 'QQQ_minute_diff', 'QQQ_gap_flag', 'QQQ_valid_row']...
ETFs found: ['GDX', 'GLD', 'QQQ', 'SHV', 'SPY', 'TLT', 'VWO', 'XLB', 'XLE', 'XME']


In [4]:
def extract_etf_data(df_bundle: pd.DataFrame, symbol: str) -> pd.DataFrame:
    df = pd.DataFrame(index=df_bundle.index)
    
    df[g_open_col] = df_bundle[f'{symbol}_S_open_f32']
    df[g_high_col] = df_bundle[f'{symbol}_S_high_f32']
    df[g_low_col] = df_bundle[f'{symbol}_S_low_f32']
    df[g_close_col] = df_bundle[f'{symbol}_S_close_f32']
    df[g_volume_col] = df_bundle[f'{symbol}_S_volume_f64']
    
    if 'i_minute_i' in df_bundle.columns:
        df[g_index_col] = df_bundle['i_minute_i']
    
    return df.dropna()

def run_rsi_backtest(
    df: pd.DataFrame,
    rsi_period: int,
    oversold: int,
    overbought: int,
    ema200_period: int,
    atr_mult: float,
    rr: float,
    direction: str = "both",
) -> Dict:
    df = build_features(
        df,
        p_direction=direction,
        p_rsi_period=rsi_period,
        p_oversold=oversold,
        p_overbought=overbought,
        p_ema200_period=ema200_period,
    )
    
    ohlcv_df = convert_to_ohlcv(df)
    
    if len(ohlcv_df) == 0:
        return None
    
    bt = Backtest(
        ohlcv_df,
        RSIMeanReversionStrategy,
        cash=100_000,
        commission=0.0002,
        trade_on_close=True,
        hedging=False,
        exclusive_orders=False,
        finalize_trades=True,
    )
    
    stats = bt.run(
        atr_mult=atr_mult,
        rr=rr,
        direction=direction,
    )
    
    return {
        'stats': stats,
        'return_pct': stats['Return [%]'],
        'sharpe': stats['Sharpe Ratio'],
        'max_dd': stats['Max. Drawdown [%]'],
        'win_rate': stats['Win Rate [%]'],
        'n_trades': stats['# Trades'],
        'profit_factor': stats['Profit Factor'],
    }

In [5]:
RSI_PERIODS = [10, 14, 21]
OVERSOLD_LEVELS = [25, 30, 35]
OVERBOUGHT_LEVELS = [65, 70, 75]
EMA200_PERIODS = [100, 150, 200]
ATR_MULTS = [1.5, 2.0, 2.5]
RR_RATIOS = [1.5, 2.0, 2.5]

total_combinations = len(RSI_PERIODS) * len(OVERSOLD_LEVELS) * len(OVERBOUGHT_LEVELS) * len(EMA200_PERIODS) * len(ATR_MULTS) * len(RR_RATIOS)
print(f"Total parameter combinations: {total_combinations}")

Total parameter combinations: 729


In [6]:
from itertools import product

def optimize_etf(
    df_bundle: pd.DataFrame,
    symbol: str,
    max_rows: int = None,
    max_runs: int = None,
    direction: str = "both",
) -> Tuple[pd.DataFrame, Dict]:
    df = extract_etf_data(df_bundle, symbol)
    
    if max_rows:
        df = df.iloc[:max_rows]
    
    print(f"{symbol}: {len(df)} rows")
    
    results = []
    
    param_combinations = list(product(
        RSI_PERIODS, OVERSOLD_LEVELS, OVERBOUGHT_LEVELS,
        EMA200_PERIODS, ATR_MULTS, RR_RATIOS
    ))
    
    if max_runs:
        param_combinations = param_combinations[:max_runs]
    
    total = len(param_combinations)
    
    for idx, (rsi_period, oversold, overbought, ema200, atr_mult, rr) in enumerate(param_combinations):
        if idx % 50 == 0:
            print(f"  Progress: {idx}/{total}")
        
        try:
            result = run_rsi_backtest(
                df,
                rsi_period=rsi_period,
                oversold=oversold,
                overbought=overbought,
                ema200_period=ema200,
                atr_mult=atr_mult,
                rr=rr,
                direction=direction,
            )
            
            if result:
                results.append({
                    'rsi_period': rsi_period,
                    'oversold': oversold,
                    'overbought': overbought,
                    'ema200': ema200,
                    'atr_mult': atr_mult,
                    'rr': rr,
                    'sharpe': result['sharpe'],
                    'return_pct': result['return_pct'],
                    'max_dd': result['max_dd'],
                    'win_rate': result['win_rate'],
                    'n_trades': result['n_trades'],
                    'profit_factor': result['profit_factor'],
                })
        except Exception as e:
            pass
    
    print(f"  Completed: {len(results)} valid results")
    
    results_df = pd.DataFrame(results)
    
    if len(results_df) == 0:
        return None, None
    
    best_idx = results_df['sharpe'].idxmax()
    best_result = results_df.loc[best_idx].to_dict()
    
    return results_df, best_result

In [7]:
MAX_ROWS = 20000
MAX_RUNS = 200
DIRECTION = "both"

all_results = {}
all_best = {}

for symbol in etf_symbols:
    print(f"\n{'='*60}")
    print(f"Optimizing {symbol}...")
    results_df, best = optimize_etf(df_bundle, symbol, max_rows=MAX_ROWS, max_runs=MAX_RUNS, direction=DIRECTION)
    
    if results_df is not None:
        all_results[symbol] = results_df
        all_best[symbol] = best
        print(f"Best Sharpe: {best['sharpe']:.4f}")
        print(f"Best params: RSI={best['rsi_period']}, OS={best['oversold']}, OB={best['overbought']}, EMA={best['ema200']}, ATR={best['atr_mult']}, RR={best['rr']}")
    else:
        print(f"No valid results for {symbol}")


Optimizing GDX...
GDX: 20000 rows
  Progress: 0/200


Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/19986 [00:00<?, ?bar/s]

KeyboardInterrupt: 

In [26]:

df = extract_etf_data(df_bundle, 'QQQ')
    
if 15000:
    df = df.iloc[:15000]

print(f"{symbol}: {len(df)} rows")

results = []

param_combinations = list(product(
    RSI_PERIODS, OVERSOLD_LEVELS, OVERBOUGHT_LEVELS,
    EMA200_PERIODS, ATR_MULTS, RR_RATIOS
))

for idx, (rsi_period, oversold, overbought, ema200, atr_mult, rr) in enumerate(param_combinations):
    total = len(param_combinations)

    result = run_rsi_backtest(
                df,
                rsi_period=rsi_period,
                oversold=oversold,
                overbought=overbought,
                ema200_period=ema200,
                atr_mult=atr_mult,
                rr=rr,
                direction="both",
            )

GDX: 15000 rows


Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/14986 [00:00<?, ?bar/s]

KeyboardInterrupt: 

In [ ]:
summary_data = []
for symbol, best in all_best.items():
    summary_data.append({
        'ETF': symbol,
        'RSI': int(best['rsi_period']),
        'Oversold': int(best['oversold']),
        'Overbought': int(best['overbought']),
        'EMA200': int(best['ema200']),
        'ATR_mult': best['atr_mult'],
        'RR': best['rr'],
        'Sharpe': round(best['sharpe'], 3),
        'Return %': round(best['return_pct'], 2),
        'MaxDD %': round(best['max_dd'], 2),
        'WinRate %': round(best['win_rate'], 2),
        '# Trades': int(best['n_trades']),
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Sharpe', ascending=False)

print("\n" + "="*100)
print("RSI MEAN-REVERSION OPTIMIZATION SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))
summary_df

In [ ]:
def plot_heatmap_rsi_oversold(results_df: pd.DataFrame, symbol: str):
    pivot = results_df.pivot_table(
        values='sharpe',
        index='oversold',
        columns='rsi_period',
        aggfunc='mean'
    )
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        cbar_kws={'label': 'Sharpe Ratio'}
    )
    plt.title(f'{symbol} - RSI Period vs Oversold Level')
    plt.xlabel('RSI Period')
    plt.ylabel('Oversold Level')
    plt.tight_layout()
    plt.show()

for symbol in list(all_results.keys())[:3]:
    plot_heatmap_rsi_oversold(all_results[symbol], symbol)

In [ ]:
def plot_heatmap_atr_rr(results_df: pd.DataFrame, symbol: str):
    pivot = results_df.pivot_table(
        values='sharpe',
        index='atr_mult',
        columns='rr',
        aggfunc='mean'
    )
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        cbar_kws={'label': 'Sharpe Ratio'}
    )
    plt.title(f'{symbol} - ATR Multiplier vs Risk-Reward')
    plt.xlabel('Risk-Reward Ratio')
    plt.ylabel('ATR Multiplier')
    plt.tight_layout()
    plt.show()

for symbol in list(all_results.keys())[:3]:
    plot_heatmap_atr_rr(all_results[symbol], symbol)

In [ ]:
def run_best_strategy(df_bundle: pd.DataFrame, symbol: str, best_params: Dict):
    df = extract_etf_data(df_bundle, symbol)
    
    df = build_features(
        df,
        p_direction=DIRECTION,
        p_rsi_period=int(best_params['rsi_period']),
        p_oversold=int(best_params['oversold']),
        p_overbought=int(best_params['overbought']),
        p_ema200_period=int(best_params['ema200']),
    )
    
    ohlcv_df = convert_to_ohlcv(df)
    
    bt = Backtest(
        ohlcv_df,
        RSIMeanReversionStrategy,
        cash=100_000,
        commission=0.0002,
        trade_on_close=True,
        hedging=False,
        exclusive_orders=False,
        finalize_trades=True,
    )
    
    stats = bt.run(
        atr_mult=best_params['atr_mult'],
        rr=best_params['rr'],
        direction=DIRECTION,
    )
    
    return bt, stats

if len(all_best) > 0:
    top_etf = summary_df.iloc[0]['ETF']
    print(f"\nRunning best strategy for {top_etf}...")
    bt, stats = run_best_strategy(df_bundle, top_etf, all_best[top_etf])
    print(stats)

In [ ]:
if len(all_best) > 0:
    bt.plot()

In [ ]:
if len(all_best) > 0:
    print("\n" + "="*100)
    print("TOP 3 ETFs - DETAILED COMPARISON")
    print("="*100)

    for idx, row in summary_df.head(3).iterrows():
        symbol = row['ETF']
        best = all_best[symbol]
        
        print(f"\n{symbol}:")
        print(f"  Parameters: RSI={int(best['rsi_period'])}, Oversold={int(best['oversold'])}, Overbought={int(best['overbought'])}")
        print(f"             EMA200={int(best['ema200'])}, ATR_mult={best['atr_mult']}, RR={best['rr']}")
        print(f"  Performance: Return={best['return_pct']:.2f}%, Sharpe={best['sharpe']:.3f}, MaxDD={best['max_dd']:.2f}%")
        print(f"  Trades: {int(best['n_trades'])} trades, Win Rate={best['win_rate']:.2f}%, Profit Factor={best['profit_factor']:.2f}")

In [ ]:
if len(summary_df) > 0:
    output_path = Path('notebooks/results/rsi_mr_optimization.csv')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")